In [78]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM, Dropout, BatchNormalization
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import ta  # 기술적 분석 라이브러리
from sklearn.metrics import mean_squared_error

In [79]:
# 데이터 전처리
def preprocess_data(df):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)
    return scaled_data

In [80]:
# 기술적 지표 생성
def create_technical_indicators(df):
    df['SMA'] = df['Close'].rolling(window=14).mean()
    df['RSI'] = ta.momentum.RSIIndicator(df['Close'], window=14).rsi()
    df['MACD'] = ta.trend.MACD(df['Close']).macd()
    df['Stochastic'] = ta.momentum.StochasticOscillator(df['High'], df['Low'], df['Close']).stoch()
    df = df.dropna()  # 결측치 제거
    return df

# 데이터 전처리
def preprocess_data(df):
    scaler = MinMaxScaler()
    scaled_data = scaler.fit_transform(df)
    
    X, y = [], []
    window_size = 20
    for i in range(len(scaled_data) - window_size):
        X.append(scaled_data[i:i + window_size])
        y.append(scaled_data[i + window_size, 3])  # 'Close' 가격 예측
    
    X, y = np.array(X), np.array(y)
    return train_test_split(X, y, test_size=0.2, random_state=0)

In [81]:
# 모델 생성
def build_model(input_shape):
    model = Sequential()
    model.add(LSTM(100, return_sequences=True, input_shape=input_shape))
    model.add(Dropout(0.3))
    model.add(BatchNormalization())
    model.add(LSTM(100, return_sequences=True))
    model.add(Dropout(0.3))
    model.add(BatchNormalization())
    model.add(LSTM(100))
    model.add(Dropout(0.3))
    model.add(Dense(1))
    model.compile(optimizer='adam', loss='mse')
    return model


In [82]:
# 매매 전략
def trading_strategy(model, X_test, y_test, threshold=0.01, stop_loss=0.05, initial_balance=1000000):
    predictions = model.predict(X_test)
    actions = []
    balance = initial_balance
    positions = 0
    for i in range(len(predictions) - 1):
        if predictions[i + 1] > predictions[i] * (1 + threshold) and balance >= y_test[i]:
            actions.append("Buy")
            positions += balance / y_test[i]
            balance = 0
        elif predictions[i + 1] < predictions[i] * (1 - threshold) and positions > 0:
            actions.append("Sell")
            balance += positions * y_test[i]
            positions = 0
        else:
            actions.append("Hold")
        
        # 손절매 전략 적용
        if actions[-1] == "Buy" and predictions[i + 1] < y_test[i] * (1 - stop_loss):
            actions[-1] = "Sell"
            balance += positions * y_test[i]
            positions = 0
        elif actions[-1] == "Sell" and predictions[i + 1] > y_test[i] * (1 + stop_loss):
            actions[-1] = "Buy"
            positions += balance / y_test[i]
            balance = 0

    if positions > 0:
        balance += positions * y_test[-1]

    return actions, balance

In [83]:
# 모델 평가
def evaluate_model(model, X_test, y_test):
    predictions = model.predict(X_test)
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    return rmse

In [84]:
# 메인 함수
def main():
    df = pd.read_csv('./datas/samsung.csv')
    df = df[['Open', 'High', 'Low', 'Close', 'Volume']]
    df = create_technical_indicators(df)
    X_train, X_test, y_train, y_test = preprocess_data(df)
    model = build_model((X_train.shape[1], X_train.shape[2]))
    
    # 모델 학습
    model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.1)

    # 매매 전략 적용 및 백테스팅
    actions, final_balance = trading_strategy(model, X_test, y_test)
    print(f"Actions: {actions}")
    print(f"Final Balance: {final_balance}")

    # 모델 평가
    rmse = evaluate_model(model, X_test, y_test)
    print(f"RMSE: {rmse}")

In [85]:
if __name__ == "__main__":
    main()

Epoch 1/100
5/5 [==============================] - 5s 299ms/step - loss: 0.1993 - val_loss: 0.1570
Epoch 2/100
5/5 [==============================] - 0s 29ms/step - loss: 0.1413 - val_loss: 0.0984
Epoch 3/100
5/5 [==============================] - 0s 21ms/step - loss: 0.0825 - val_loss: 0.1456
Epoch 4/100
5/5 [==============================] - 0s 21ms/step - loss: 0.0517 - val_loss: 0.1325
Epoch 5/100
5/5 [==============================] - 0s 19ms/step - loss: 0.0394 - val_loss: 0.1239
Epoch 6/100
5/5 [==============================] - 0s 20ms/step - loss: 0.0334 - val_loss: 0.1103
Epoch 7/100
5/5 [==============================] - 0s 22ms/step - loss: 0.0279 - val_loss: 0.1106
Epoch 8/100
5/5 [==============================] - 0s 22ms/step - loss: 0.0276 - val_loss: 0.0809
Epoch 9/100
5/5 [==============================] - 0s 20ms/step - loss: 0.0205 - val_loss: 0.0900
Epoch 10/100
5/5 [==============================] - 0s 20ms/step - loss: 0.0199 - val_loss: 0.0832
Epoch 11/100
5/5 [